In [ ]:
!pip install pycirclize

In [ ]:
from pycirclize import Circos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import glob
import os

# Load multiple AlphaMissense CSVs
csv_folder = "AlphaMissense/"
csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))

# Map protein names to sector lengths
sectors = {
    "TSC22D1": 1073, "TSC22D2": 780, "TSC22D3": 134, "TSC22D4": 395,
    "WNK1": 2382, "WNK2": 2297, "WNK3": 1800, "WNK4": 1243,
    "NRBP1": 535, "NRBP2": 501, "STK39": 545, "OXSR1": 527
}
# Dummy sector (for spacing/adjustment)
sectors["DUMMY"] = 50

# Build data matrices for each protein
aa_order = list("ACDEFGHIKLMNPQRSTVWY")
protein_matrices = {}

for file in csv_files:
    protein_name = os.path.basename(file).split("_")[0]
    if protein_name not in sectors:
        continue
    df = pd.read_csv(file)
    pivot = df.pivot_table(index="AA", columns="Residue",
                           values="Pathogenicity", aggfunc="mean")
    pivot = pivot.reindex(index=aa_order).fillna(0)
    protein_matrices[protein_name] = pivot.to_numpy()

# Load ConSurf data
consurf_folder = "ConSurf/"
consurf_files = glob.glob(os.path.join(consurf_folder, "*.csv"))
consurf_matrices = {}

for file in consurf_files:
    protein_name = os.path.basename(file).split("_")[0]
    if protein_name not in sectors:
        continue
    df = pd.read_csv(file, skiprows=4)
    df = df[['pos', 'ConSurf grade']].dropna()
    df['pos'] = df['pos'].astype(int)
    df['ConSurf grade'] = pd.to_numeric(df['ConSurf grade'], errors='coerce')
    scores = np.zeros(sectors[protein_name])
    scores[df['pos'] - 1] = df['ConSurf grade']
    consurf_matrices[protein_name] = scores

# Load Phosphosite data
phosphosites_folder = "Phosphosites/"
phosphosites_files = glob.glob(os.path.join(phosphosites_folder, "*.csv"))
phosphosites_matrices = {}

for file in phosphosites_files:
    protein_name = os.path.basename(file).split("_")[0]
    if protein_name not in sectors:
        continue
    df = pd.read_csv(file)
    df = df[['Residue', 'Invert_LogRank']].dropna()
    df['Residue'] = df['Residue'].astype(int)
    df['Invert_LogRank'] = pd.to_numeric(df['Invert_LogRank'], errors='coerce')
    scores = np.zeros(sectors[protein_name])
    scores[df['Residue'] - 1] = df['Invert_LogRank']
    phosphosites_matrices[protein_name] = scores

# Define custom colormaps (black at 0 + white -> color gradient)
def make_gradient(hex_color):
    gradient = mcolors.LinearSegmentedColormap.from_list(
        f"white_{hex_color}", [(0, "white"), (1, hex_color)]
    )
    colors = gradient(np.linspace(0, 1, 256))
    newcolors = np.vstack((np.array([0, 0, 0, 1]), colors))  # add black at index 0
    return mcolors.ListedColormap(newcolors)

alphamiss_cmaps = {
    "TSC22D1": make_gradient("#ffe5b6"),
    "TSC22D2": make_gradient("#ffe5b6"),
    "TSC22D3": make_gradient("#ffe5b6"),
    "TSC22D4": make_gradient("#ffe5b6"),
    "WNK1": make_gradient("#49c1bb"),
    "WNK2": make_gradient("#49c1bb"),
    "WNK3": make_gradient("#49c1bb"),
    "WNK4": make_gradient("#49c1bb"),
    "NRBP1": make_gradient("#bababa"),
    "NRBP2": make_gradient("#bababa"),
    "STK39": make_gradient("#6dabc6"),
    "OXSR1": make_gradient("#6dabc6"),
}

# ConSurf colormap: white → gray
consurf_cmap = mcolors.LinearSegmentedColormap.from_list(
    "white_#b3b3b3", [(0, "white"), (1, "#b3b3b3")]
)

# Build Circos object
circos = Circos(sectors, space=2)

# Chain → Protein mapping
chain_to_protein = {
    "t1": "TSC22D1",
    "t2": "TSC22D2",
    "t3": "TSC22D3",
    "t4": "TSC22D4",
    "w1": "WNK1",
    "w2": "WNK2",
    "w3": "WNK3",
    "w4": "WNK4",
    "n1": "NRBP1",
    "n2": "NRBP2",
    "s": "STK39",
    "o": "OXSR1",
}

# Add residue-residue links
links_file = "Figure_5C_Result_with_Category.csv"
df_links = pd.read_csv(links_file)

# Define category-based styles
category_styles = {
    "Intersection": {"color": "#bababa", "lw": 0.1, "alpha": 0.3, "linestyle": "solid"},
    "Only_in_File1": {"color": "#ffe5b6", "lw": 0.1, "alpha": 0.3, "linestyle": "solid"},
    "Only_in_File2": {"color": "#49c1bb", "lw": 0.1, "alpha": 0.3, "linestyle": "solid"},
}

# Add links between interacting residues
for _, row in df_links.iterrows():
    prot1 = chain_to_protein.get(str(row["Chain 1"]).strip(), None)
    prot2 = chain_to_protein.get(str(row["Chain 2"]).strip(), None)
    res1, res2, cat = row["Residue 1"], row["Residue 2"], row["Category"]

    if prot1 not in sectors or prot2 not in sectors:
        continue

    style = category_styles.get(cat, {"color": "black", "lw": 0.3, "alpha": 0.3, "linestyle": "solid"})

    circos.link(
        (prot1, int(res1), int(res1)),
        (prot2, int(res2), int(res2)),
        r1=40,
        r2=40,
        color=style["color"],
        lw=style["lw"],
        alpha=style["alpha"],
        linestyle=style["linestyle"]
    )

# Hide Dummy sector
dummy_sector = circos.sectors[-1]
dummy_sector.visible = False

# Add multiple data tracks for each protein
for sector in circos.sectors:
    if sector.name == "DUMMY":
        continue
    protein = sector.name

    # Track 1: AlphaMissense heatmap
    track1 = sector.add_track((40, 82))
    track1.axis()
    if protein in protein_matrices:
        cmap = alphamiss_cmaps.get(protein)
        track1.heatmap(
            protein_matrices[protein],
            vmin=1e-11,
            vmax=1,
            cmap=cmap
        )
    if protein == "TSC22D1":
        track1.yticks(
            np.arange(0.5, len(aa_order)),
            aa_order[::-1],
            vmin=0,
            vmax=len(aa_order),
            side="left",
            line_kws=dict(lw=0.5),
            text_kws=dict(size=1.5, fontfamily="Arial")
        )

    # Track 2: ConSurf conservation scores
    track2 = sector.add_track((83, 86))
    if protein in consurf_matrices:
        track2.heatmap(
            consurf_matrices[protein][np.newaxis, :],
            vmin=1,
            vmax=9,
            cmap=consurf_cmap,
            rect_kws=dict(ec="none")
        )

    # Track 3: Phosphosite bar plot
    track3 = sector.add_track((87, 94))
    track3.axis(ec="none", fc="none")
    if protein in phosphosites_matrices:
        scores = phosphosites_matrices[protein]
        x = np.arange(1, len(scores) + 1)
        track3.bar(
            x,
            scores,
            width=4,
            facecolor="black",
            edgecolor="none",
        )
    if protein == "TSC22D1":
        track3.yticks(
            [0, 1, 2],
            ["0", "1", "2"],
            side="left",
            line_kws=dict(lw=0.5),
            text_kws=dict(size=1.5, fontfamily="Arial")
        )

    # Track 4: Sector labels
    track4 = sector.add_track((95, 100))
    track4.text(
        sector.name,
        sector.size / 2,
        ha="center",
        va="center",
        fontfamily="Arial",
        size=8,
        adjust_rotation=True,
        orientation="horizontal"
    )

# Plot & Save Figure
fig = circos.plotfig()

# Rasterize large heatmap elements for smaller PDF size
for ax in fig.axes:
    for artist in ax.get_children():
        if artist.__class__.__name__ in ["QuadMesh", "PolyCollection"]:
            artist.set_rasterized(True)

# Save outputs
fig.savefig("Figure_5C_Circos_Tetramer.pdf", format="pdf",
            dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("Figure_5C_Circos_Tetramer.png", format="png",
            dpi=1500, bbox_inches="tight", transparent=True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Define gradient colormap function (same as main plot)
def make_gradient(hex_color):
    """Colormap: black at 0, white → given color"""
    gradient = mcolors.LinearSegmentedColormap.from_list(
        f"white_{hex_color}", [(0, "white"), (1, hex_color)]
    )
    colors = gradient(np.linspace(0, 1, 256))
    newcolors = np.vstack((np.array([0, 0, 0, 1]), colors))  # Add black at index 0
    return mcolors.ListedColormap(newcolors)

# AlphaMissense
alphamiss_TSC22D_cmap = make_gradient("#ffe5b6")
alphamiss_WNK_cmap = make_gradient("#49c1bb")
alphamiss_NRBP_cmap = make_gradient("#bababa")
alphamiss_SO_cmap = make_gradient("#6dabc6")

# ConSurf colormap
consurf_cmap = mcolors.LinearSegmentedColormap.from_list(
    "white_gray", [(0, "white"), (1, "#b3b3b3")]
)

# Create figure
fig, axes = plt.subplots(1, 5, figsize=(5, 3))

# Track1: TSC22D
sm1 = plt.cm.ScalarMappable(cmap=alphamiss_TSC22D_cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar1 = fig.colorbar(sm1, cax=axes[0], orientation="vertical")
cbar1.set_label("AlphaMissense Pathogenicity\n0=Benign → 1=Pathogenic\nblack → Unaltered residue", 
                fontsize=9, fontfamily="Arial")
axes[0].set_title("Track1_TSC22D", fontsize=10, fontfamily="Arial")

# Track1: AlphaMissense heatmap style
sm1 = plt.cm.ScalarMappable(cmap=alphamiss_WNK_cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar1 = fig.colorbar(sm1, cax=axes[1], orientation="vertical")
cbar1.set_label("AlphaMissense Pathogenicity\n0=Benign → 1=Pathogenic\nblack → Unaltered residue", 
                fontsize=9, fontfamily="Arial")
axes[1].set_title("Track1_WNK", fontsize=10, fontfamily="Arial")

# Track1: AlphaMissense heatmap style
sm1 = plt.cm.ScalarMappable(cmap=alphamiss_NRBP_cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar1 = fig.colorbar(sm1, cax=axes[2], orientation="vertical")
cbar1.set_label("AlphaMissense Pathogenicity\n0=Benign → 1=Pathogenic\nblack → Unaltered residue", 
                fontsize=9, fontfamily="Arial")
axes[2].set_title("Track1_NRBP", fontsize=10, fontfamily="Arial")

# Track1: AlphaMissense heatmap style
sm1 = plt.cm.ScalarMappable(cmap=alphamiss_SO_cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar1 = fig.colorbar(sm1, cax=axes[3], orientation="vertical")
cbar1.set_label("AlphaMissense Pathogenicity\n0=Benign → 1=Pathogenic\nblack → Unaltered residue", 
                fontsize=9, fontfamily="Arial")
axes[3].set_title("Track1_SO", fontsize=10, fontfamily="Arial")

# Track2: ConSurf heatmap style
sm2 = plt.cm.ScalarMappable(cmap=consurf_cmap, norm=plt.Normalize(vmin=1, vmax=9))
cbar2 = fig.colorbar(sm2, cax=axes[4], orientation="vertical", ticks=range(1, 10))
cbar2.set_label("ConSurf Grade\n1=Variable → 9=Conserved", 
                fontsize=9, fontfamily="Arial")
axes[4].set_title("Track2", fontsize=10, fontfamily="Arial")

# Layout & Save
plt.tight_layout()
plt.subplots_adjust(wspace=9)
fig.savefig("Figure_5C_Circos_legends.pdf", dpi=600, bbox_inches="tight")
fig.savefig("Figure_5C_Circos_legends.png", dpi=600, bbox_inches="tight")
plt.show()
